In [ ]:
import pandas as pd
import numpy as np

In [ ]:
SETUP FOR PYTHON

In [ ]:
xl = pd.read_excel(r"C:\Users\Tanisha Iyer\Desktop\Projects\DSM Project\ocds_mapped_data_fiscal_year_2016_2022_v3.xlsx", sheet_name=None)

main          = xl['main']
awards        = xl['awards']
awards_sup    = xl['awards_suppliers']
bids          = xl['bids_details']
bids_tend     = xl['bids_details_tenderers']

# Filter to FY 2020-23 only
target_years  = ['2020-2021', '2021-2022', '2022-2023']
main_23       = main[main['tender_fiscalYear'].isin(target_years)]

# Join awards to their supplier names and parent tender
awards_full = (
    awards
    .merge(awards_sup[['_link_awards','name']], left_on='_link', right_on='_link_awards')
    .merge(main_23[['_link','buyer_name','tender_fiscalYear','tender_value_amount']],
           left_on='_link_main', right_on='_link')
    .rename(columns={'name':'supplier_name', 'value_amount':'award_value'})
)

# Drop placeholder values
awards_full = awards_full[awards_full['award_value'] > 1]

print(awards_full.shape)  # check it loaded

In [ ]:
Add Sector Labels - since no classifier 

In [ ]:
sector_map = {
    'Public Works Roads Department':              'roads',
    'Public Works Building and NH Department':    'buildings',
    'National Health Mission':                    'health',
    'Department of Water Resources':              'water_sanitation',
    'Assam Power Distribution Company Ltd':       'electricity',
    'Guwahati Municipal Corporation':             'buildings',
    'Bodoland Territorial Council':               'buildings',
    'Bodoland Territorial Council-PWD':           'roads',
    'Industries and Commerce Department':         'other',
    'Home B':                                     'other',
}

awards_full['sector'] = awards_full['buyer_name'].map(sector_map).fillna('other')

In [ ]:
HHI by Sector and Buyer 

In [ ]:
def compute_hhi(group):
    total = group['award_value'].sum()
    shares = group.groupby('supplier_name')['award_value'].sum() / total
    return (shares ** 2).sum() * 10000

# HHI by sector
hhi_sector = (
    awards_full
    .groupby('sector')
    .apply(compute_hhi)
    .reset_index()
    .rename(columns={0: 'HHI'})
    .sort_values('HHI', ascending=False)
)
print(hhi_sector)

# HHI by buyer
hhi_buyer = (
    awards_full
    .groupby('buyer_name')
    .apply(compute_hhi)
    .reset_index()
    .rename(columns={0: 'HHI'})
    .sort_values('HHI', ascending=False)
)
print(hhi_buyer.head(20))

In [ ]:
Gini Coefficient and Lorenz Curve 

In [ ]:
import matplotlib.pyplot as plt

def gini(values):
    arr = np.sort(np.array(values, dtype=float))
    n = len(arr)
    index = np.arange(1, n + 1)
    return (2 * (index * arr).sum()) / (n * arr.sum()) - (n + 1) / n

def lorenz(values):
    arr = np.sort(np.array(values, dtype=float))
    scaled = arr.cumsum() / arr.sum()
    return np.concatenate([[0], scaled])

# Overall Gini
supplier_totals = awards_full.groupby('supplier_name')['award_value'].sum()
print(f"Overall Gini: {gini(supplier_totals):.3f}")

# Lorenz curve per sector
sectors = awards_full['sector'].unique()
fig, ax = plt.subplots(figsize=(10, 7))

for s in sectors:
    vals = (awards_full[awards_full['sector'] == s]
            .groupby('supplier_name')['award_value'].sum())
    lc = lorenz(vals)
    x = np.linspace(0, 1, len(lc))
    ax.plot(x, lc, label=f"{s} (Gini={gini(vals):.2f})")

# Perfect equality line
ax.plot([0,1], [0,1], 'k--', label='Perfect equality')
ax.set_xlabel('Cumulative share of suppliers')
ax.set_ylabel('Cumulative share of award value')
ax.set_title('Lorenz Curves by Sector — Assam Procurement FY 2020–23')
ax.legend()
plt.tight_layout()
plt.savefig('lorenz_curves.png', dpi=150)
plt.show()

In [ ]:
CR4 and CR10 Concentration Ratios

In [ ]:
def concentration_ratio(group, n):
    total = group['award_value'].sum()
    top_n = group.groupby('supplier_name')['award_value'].sum().nlargest(n).sum()
    return top_n / total * 100

results = []
for s in awards_full['sector'].unique():
    grp = awards_full[awards_full['sector'] == s]
    results.append({
        'sector': s,
        'CR4':  concentration_ratio(grp, 4),
        'CR10': concentration_ratio(grp, 10),
        'HHI':  compute_hhi(grp)
    })

cr_df = pd.DataFrame(results).sort_values('CR4', ascending=False)
print(cr_df)

In [ ]:
HHI Heatmap (Sector × Buyer)

In [ ]:
import seaborn as sns

# Compute HHI for each buyer × sector combination
hhi_matrix = {}
for (buyer, sector), grp in awards_full.groupby(['buyer_name', 'sector']):
    if len(grp) >= 3:  # only cells with enough data
        hhi_matrix[(buyer, sector)] = compute_hhi(grp)

hhi_df = pd.Series(hhi_matrix).reset_index()
hhi_df.columns = ['buyer', 'sector', 'HHI']

# Pivot to matrix
pivot = hhi_df.pivot(index='buyer', columns='sector', values='HHI')

# Keep only top 20 buyers by tender count for readability
top_buyers = main_23['buyer_name'].value_counts().head(20).index
pivot = pivot[pivot.index.isin(top_buyers)]

plt.figure(figsize=(14, 10))
sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': 'HHI (higher = more concentrated)'})
plt.title('HHI Heatmap — Sector × Buyer  |  Assam FY 2020–23')
plt.tight_layout()
plt.savefig('hhi_heatmap.png', dpi=150)
plt.show()

In [ ]:
Bipartite Buyer-Supplier Network

In [ ]:
import networkx as nx

# Build the graph
G = nx.Graph()

for _, row in awards_full.iterrows():
    buyer    = f"B:{row['buyer_name']}"
    supplier = f"S:{row['supplier_name']}"
    if G.has_edge(buyer, supplier):
        G[buyer][supplier]['weight'] += row['award_value']
    else:
        G.add_edge(buyer, supplier, weight=row['award_value'])

# Tag node types
for n in G.nodes():
    G.nodes[n]['type'] = 'buyer' if n.startswith('B:') else 'supplier'

print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

# Degree (how many connections each node has)
degree = pd.Series(dict(G.degree())).sort_values(ascending=False)
print("Top 10 most connected:")
print(degree.head(10))

# Identify hub suppliers (connected to many buyers)
supplier_nodes = [n for n in G.nodes() if n.startswith('S:')]
hub_suppliers = (
    pd.Series({n: len(list(G.neighbors(n))) for n in supplier_nodes})
    .sort_values(ascending=False)
    .head(20)
)
print("Hub suppliers (most buyers served):")
print(hub_suppliers)

In [ ]:
Visualisation

In [ ]:
# Keep only top 15 buyers and top 30 suppliers by degree
top_b = [n for n in degree.index if n.startswith('B:')][:15]
top_s = [n for n in degree.index if n.startswith('S:')][:30]
subG  = G.subgraph(top_b + top_s)

colors = ['#2E75B6' if n.startswith('B:') else '#E74C3C'
          for n in subG.nodes()]
sizes  = [3000 if n.startswith('B:') else 1000
          for n in subG.nodes()]

plt.figure(figsize=(16, 12))
pos = nx.spring_layout(subG, seed=42, k=2)
nx.draw_networkx(subG, pos,
                 node_color=colors, node_size=sizes,
                 font_size=7, edge_color='#CCCCCC',
                 labels={n: n[2:] for n in subG.nodes()})
plt.title('Buyer–Supplier Network  |  Top 15 Buyers, Top 30 Suppliers')
plt.axis('off')
plt.tight_layout()
plt.savefig('network.png', dpi=150)
plt.show()

In [ ]:
 K-Means Clustering of Buyers

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Build feature vector per buyer
buyer_features = []

for buyer, grp in awards_full.groupby('buyer_name'):
    main_grp = main_23[main_23['buyer_name'] == buyer]
    
    # Single bidder rate
    n_tend = main_grp['tender_numberOfTenderers'].dropna()
    sbr    = (n_tend == 1).mean() if len(n_tend) > 0 else 0
    
    buyer_features.append({
        'buyer':            buyer,
        'mean_value':       grp['award_value'].mean(),
        'median_bidders':   main_grp['tender_numberOfTenderers'].median(),
        'single_bidder_rt': sbr,
        'hhi':              compute_hhi(grp),
        'award_count':      len(grp),
    })

feat_df = pd.DataFrame(buyer_features).set_index('buyer')
feat_df = feat_df.dropna()

# Standardise
scaler = StandardScaler()
X      = scaler.fit_transform(feat_df)

# K-Means with 4 clusters
km = KMeans(n_clusters=4, random_state=42, n_init=10)
feat_df['cluster'] = km.fit_predict(X)

# Label clusters meaningfully based on their characteristics
print(feat_df.groupby('cluster').mean().round(2))
print()
print("Buyers per cluster:")
for c in range(4):
    print(f"\nCluster {c}:")
    print(feat_df[feat_df['cluster']==c].index.tolist())